**Running this on Google Colab.** The first code cell fetches this entry's files and installs the packages it was run with, pinned for Colab's Python version. The results in the registry come from the entry's own environment ([requirements.lock.txt](requirements.lock.txt), Python 3.13); a different Python or platform can change the last digits of some numbers.

In [ ]:
# On Google Colab: fetch this entry's files and install its pinned packages. Elsewhere this cell does nothing.
import sys
if "google.colab" in sys.modules:
    import os, subprocess
    if not os.path.isdir("/content/encore"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", "colab", "--filter=blob:none", "--sparse",
                        "https://github.com/encorescience/encore-registry", "/content/encore"], check=True)
        subprocess.run(["git", "-C", "/content/encore", "sparse-checkout", "set", "entries/prakash-2022"], check=True)
    os.chdir("/content/encore/entries/prakash-2022")
    sys.path.insert(0, os.getcwd())
    version = f"{sys.version_info.major}.{sys.version_info.minor}"
    lock = f"requirements.colab-{version}.txt"
    if not os.path.exists(lock):
        print(f"No lock for Python {version}: installing the entry's unpinned requirements (requirements.in).")
        lock = "requirements.in"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    subprocess.run(["uv", "pip", "install", "--system", "-q", "-r", lock], check=True)
    print("Installed. If Colab asks to restart the session, restart it and run the notebook from the top again.")


# Reproducing Prakash et al. (2022) from dandiset 000935 and the article's source data

Paper: Prakash, Murphy, St Laurent, Friedman, Crespo, Bjorefeldt, Pal, Bhagat, Kauer, Shaner,
Lipscombe, Moore and Hochgeschwender (2022), Selective control of synaptically-connected circuit
elements by all-optical synapses, Communications Biology 5, 33,
[doi:10.1038/s42003-021-02981-7](https://doi.org/10.1038/s42003-021-02981-7).

Dataset: [DANDI:000935](https://dandiarchive.org/dandiset/000935/0.240319.2026), version
`0.240319.2026` (doi:10.48324/dandi.000935/0.240319.2026). The paper cites no version (the
dandiset was created in 2024, two years after the paper). This is the only published version.
The draft holds the same six files (the same blobs, checked below); only its metadata differ,
so the published version is pinned.

The claims file (`claims/prakash-2022.yaml`) was written from the version of record (PMC8752598),
with the supplementary PDFs. The dandiset holds only the in vivo experiment (Fig. 7): six PV-Cre
mice, 32-contact laminar recordings in barrel cortex and bioluminescence imaging (none for one
Opsin (+) mouse). The in vitro claims (Figs. 2 to 6) are marked `data: elsewhere` in the claims
file: they are tested on the per-electrode spike counts in Supplementary Data 1 to 6 of the
article, read from the publisher's site.

| Claim | Data | Tested here |
|---|---|---|
| fig2d-chr2-ctz-excites | Supplementary Data 1 | tested |
| fig2d-gtacr2-ctz-inhibits | Supplementary Data 1 | tested |
| fig3c-uncut-responses | Supplementary Data 2 | tested |
| fig3c-cut-abolishes-ctz | Supplementary Data 2 | tested |
| figs4c-blockers-silence | Supplementary Data 3 | tested |
| fig4c-sb-ctz-activates | Supplementary Data 4 | tested |
| fig4e-late-ctz-no-effect | Supplementary Data 4 | tested |
| fig4e-stim-then-ctz-activates | Supplementary Data 4 | tested |
| fig4e-stim-alone-no-effect | Supplementary Data 4 | tested |
| fig5c-bont-blocks | Supplementary Data 5 | tested |
| figs5-bont-reduces-release | IVIS images, not deposited | blocked (data missing) |
| figs6-luciferase-in-dcvs | confocal images, not deposited | blocked (data missing) |
| fig6c-green-light-abolishes | Supplementary Data 6 | tested |
| fig6f-nonfunctional-opsin-no-response | Supplementary Data 6 | tested |
| figs3-ctz-acts-postsynaptically | MEA recordings, not deposited | blocked (data missing) |
| figs2-cut-disrupts-synchrony | MEA recordings, not deposited | blocked (data missing) |
| fig7b to fig7f (10 claims) | dandiset 000935 | tested |

The in vivo computation streams the recordings three times (once for the errant contacts and the LFP, once for each kept-contact set); the six files hold 49.1 GB with the images. The in vivo pipeline step took 38 to 50 minutes in four earlier runs (this run's time is printed below it), and the whole notebook 46 to 51 minutes in the last three earlier runs.

In [ ]:
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
from scipy import stats

import interlum as il

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 200)
T0 = time.time()
print("dandiset", il.DANDISET_ID, "version", il.DANDISET_VERSION)

## Data check

What the files hold, and each description checked against the values.

In [ ]:
# The six files at the pinned version, and the draft's files (same blobs means same content).
api = f"https://api.dandiarchive.org/api/dandisets/{il.DANDISET_ID}/versions"
pub = requests.get(f"{api}/{il.DANDISET_VERSION}/assets/?page_size=100", timeout=60).json()["results"]
dra = requests.get(f"{api}/draft/assets/?page_size=100", timeout=60).json()["results"]
print("published:", len(pub), "files,", round(sum(a["size"] for a in pub) / 1e9, 1), "GB")
print("draft files with the same path and blob:",
      len({(a["path"], a["blob"]) for a in pub} & {(a["path"], a["blob"]) for a in dra}), "of", len(dra))

sess = il.sessions()
GROUP = {s.subject: s.group for s in sess}
PLUS = [s.subject for s in sess if s.group == "Opsin (+)"]
MINUS = [s.subject for s in sess if s.group == "Opsin (-)"]
rows = []
for s in sess:
    rows.append(dict(subject=s.subject, group=s.group, age=s.age, sex=s.sex,
                     ephys_rate_Hz=s.fs, ephys_s=round(s.duration, 1), channels=s.n_channels,
                     unit=s.ephys_unit, intervals=s.n_intervals,
                     infusion_start=s.infusion[0], infusion_stop=s.infusion[1],
                     image_frames=s.image_shape[0] if s.has_imaging else None,
                     image_px=f"{s.image_shape[1]}x{s.image_shape[2]}" if s.has_imaging else None,
                     image_rate_Hz=s.image_rate if s.has_imaging else None))
meta = pd.DataFrame(rows).set_index("subject")
print(meta.to_string())
print()
print("groups:", {g: sum(1 for s in sess if s.group == g) for g in ("Opsin (+)", "Opsin (-)")})
print("ephys description:", sorted({s.ephys_description for s in sess}))
print("ephys and image series start at file time 0 in every file:",
      all(s.ephys_t0 == 0 for s in sess), all(s.image_t0 == 0 for s in sess if s.has_imaging))
print("infusion lasts", sorted({s.infusion[1] - s.infusion[0] for s in sess}), "s;",
      "50 ul at 25 ul/min in the Methods is 120 s")
pos = sess[0].position
print("electrode Position is the same in every file:", all(np.array_equal(s.position, pos) for s in sess))
print("Position x values", sorted(set(pos[:, 0])), "| y values", sorted(set(pos[:, 1])),
      "| z from", pos[:, 2].min(), "to", pos[:, 2].max(), "in steps of", np.unique(np.diff(np.sort(pos[:, 2]))))
print("channel 0 is", sess[0].labels[0], "at z =", pos[0, 2], "; channel 31 at z =", pos[31, 2])
print()
print("Paper (Methods) against files:")
print("  sampling rate: Methods 30 kHz; file description 'acquired at 20kHz, downsampled to 10kHz'; files at",
      sorted({s.fs for s in sess}), "Hz")
print("  images: Methods 512 x 512 pixels; files", sorted({f"{s.image_shape[1]}x{s.image_shape[2]}" for s in sess if s.has_imaging}))
print("  imaging missing for one Opsin (+) mouse: files without images:",
      [s.subject + " " + s.group for s in sess if not s.has_imaging])
print("  recording after infusion start (Methods: at least 20 min):",
      {s.subject: round((s.duration - s.infusion[0]) / 60, 1) for s in sess}, "min")

In [ ]:
# Recordings deposited twice? Compare the first second of every file, and the image series' lengths.
first = {s.subject: s.ephys(0, 10_000) for s in sess}
same = [(a, b) for i, a in enumerate(first) for b in list(first)[i + 1:]
        if first[a].shape == first[b].shape and np.allclose(first[a], first[b])]
print("pairs of files with an identical first second of ephys:", same)
cc = np.corrcoef([first[k][:, 0] for k in first])
print("largest correlation of channel 0 between two files:", round(np.max(cc - np.eye(len(cc))), 3))
print("ephys lengths (samples):", {s.subject: s.n_samples for s in sess})
del first

### Bioluminescence and its onset

The Methods average a 50-pixel circular ROI placed by hand next to the electrode shank, and take the
onset as the peak of the discrete derivative. The dandiset records no ROI position, so only reading
(o2b) of the claims file, the mean over the whole frame, can be run. Frames that are entirely zero
are dropped frames; they are treated as missing.

In [ ]:
bio = {}
for s in sess:
    if s.has_imaging:
        bio[s.subject] = il.bioluminescence(s)
onset = {k: float(il.onset_index(v)) for k, v in bio.items()}
delay = {s.subject: onset[s.subject] - s.infusion[0] for s in sess if s.has_imaging}
mean_delay = float(np.mean(list(delay.values())))
rows = []
for s in sess:
    if not s.has_imaging:
        continue
    b = bio[s.subject]
    d = np.diff(b)
    top = np.argsort(np.nan_to_num(d, nan=-np.inf))[::-1][:2]
    rows.append(dict(subject=s.subject, group=s.group, frames=len(b), blank_frames=int(np.isnan(b).sum()),
                     baseline_counts=round(np.nanmean(b[:150]), 1), max_counts=round(np.nanmax(b), 1),
                     onset_s=onset[s.subject], infusion_start_s=s.infusion[0],
                     onset_minus_infusion_s=delay[s.subject],
                     largest_rise=round(d[top[0]], 1), second_rise_at=int(top[1]), second_rise=round(d[top[1]], 1),
                     post_s_imaged=min(il.POST_S, len(b) - onset[s.subject])))
bt = pd.DataFrame(rows).set_index("subject")
print(bt.to_string())
print("mean delay from infusion start to onset in the five imaged mice:", round(mean_delay, 1), "s")
blank = np.flatnonzero(np.isnan(bio["interlum_4"]))
print("blank frames of interlum_4 (s):", blank.min(), "to", blank.max(), "in", len(blank), "frames")

fig, axs = plt.subplots(1, 5, figsize=(17, 3), sharex=True)
for ax, (k, b) in zip(axs, bio.items()):
    t = np.arange(len(b)) - onset[k]
    ax.plot(t, b, lw=0.7)
    ax.axvline(0, color="k", lw=0.5)
    ax.axvline(dict((s.subject, s.infusion[0]) for s in sess)[k] - onset[k], color="r", lw=0.5)
    ax.set_title(f"{k} ({GROUP[k]})", fontsize=9)
    ax.set_xlabel("s from onset (black); red: infusion start")
axs[0].set_ylabel("whole-frame mean (counts)")
plt.tight_layout(); plt.show()
print("all points shown; x range", min(-onset[k] for k in bio), "to", max(len(b) - onset[k] for k, b in bio.items()), "s")

Clocks. Each file has one interval, the CTZ infusion, and its ephys and image series both start at
file time 0. The onset follows the infusion start in four of the five imaged mice and precedes it by
8 s in interlum_ctrl_4 (table above), so the infusion interval and the images agree in four files and
may be offset in that one. The paper's own series that best matches this mouse (Supplementary Data 7, cntrl 2, r = 0.990) puts the paper's onset 4 s after the one found here, also before the recorded infusion start. That the ephys shares the images' clock is consistent with the Methods'
TTL synchronisation but is not shown here: a contrast on the ephys around the onset would use the
very effect the claims test. interlum_6 has no images, so for it only the infusion interval is
available.

Check against the paper's own series. Supplementary Data 7 (Fig. 7b) gives the ROI series of the
five imaged mice, aligned to the paper's onset (rows from -180 to 1200 s), as "exp 1", "exp 3" and
"cntrl 1" to "cntrl 3". The dandiset does not say which of its mice these are. Each whole-frame
series is correlated with each of them, over lags of up to 120 s, to see which recordings match and
how far the onsets found here are from the paper's. This is a check on the data only; no outcome
uses it.

In [ ]:
rows7b = il.supplementary_sheet(7, "Fig. 7b")
hdr = [h for h in rows7b[4] if h is not None]
sd7 = np.array([[np.nan if v is None else v for v in r[:len(hdr)]] for r in rows7b[5:]
                if isinstance(r[0], (int, float))], float)
print("Supplementary Data 7, Fig. 7b:", sd7.shape[0], "rows from", sd7[0, 0], "to", sd7[-1, 0], "s; columns", hdr[1:])
match, match_num = {}, {}
for k, b in bio.items():
    best = {}
    for j, name in enumerate(hdr[1:], start=1):
        res = (-np.inf, None)
        for lag in range(-120, 121):
            idx = (sd7[:, 0] + onset[k] + lag).astype(int)
            ok = (idx >= 0) & (idx < len(b))
            x, y = b[idx[ok]], sd7[ok, j]
            m = ~np.isnan(x) & ~np.isnan(y)
            if m.sum() > 300:
                r = np.corrcoef(x[m], y[m])[0, 1]
                if r > res[0]:
                    res = (r, lag)
        best[name] = f"r={res[0]:.3f} lag={res[1]}"
        match_num.setdefault(k, {})[name] = res
    match[k] = best
print(pd.DataFrame(match).T.to_string())
print("A positive lag means the paper's onset is later than the one found here.")
print("last time with a value in each Supplementary Data 7 series (s after onset):",
      {h: float(sd7[~np.isnan(sd7[:, j]), 0].max()) for j, h in enumerate(hdr) if j > 0})
print("imaged time after the onset found here (s):", {k: len(b) - 1 - onset[k] for k, b in bio.items()})

Onset sensitivity. The onset is the largest rise of the whole-frame series. In three mice the second
largest rise is nearly as large and close by, and the Supplementary Data 7 match above puts the
paper's onsets up to tens of seconds away. Two other onsets are therefore run for each imaged mouse, as
a sensitivity check and not as a reading of the claims file: "second", the second largest rise where it
lies within 60 s of the largest (else the onset itself), and "sd7", the onset moved by the lag of the
best-matching Supplementary Data 7 series of the same group. The claims are judged on the onset itself;
the section "Onset sensitivity" below shows what the alternatives change.

In [ ]:
ALT = {}
for k, b in bio.items():
    d = np.diff(b)
    top = np.argsort(np.nan_to_num(d, nan=-np.inf))[::-1][:2]
    same = [h for h in hdr[1:] if h.startswith("exp" if GROUP[k] == "Opsin (+)" else "cntrl")]
    best = max(same, key=lambda h: match_num[k][h][0])
    ALT[k] = dict(onset=onset[k], second_rise_at=int(top[1]),
                  second=float(top[1]) if abs(int(top[1]) - int(top[0])) <= 60 else onset[k],
                  sd7_series=best, sd7_r=round(float(match_num[k][best][0]), 3), sd7_lag=match_num[k][best][1],
                  sd7=onset[k] + match_num[k][best][1])
print(pd.DataFrame(ALT).T.to_string())
ONSETS = {s.subject: ({"imaging": onset[s.subject], "second": ALT[s.subject]["second"], "sd7": ALT[s.subject]["sd7"]}
                      if s.has_imaging else
                      {"o1a": s.infusion[0], "o1b": s.infusion[0] + mean_delay}) for s in sess}
print("onsets used (file seconds):", ONSETS)


## The in vivo pipeline

Every measure of the claims file's "in vivo pipeline", for every session, under both kept-contact
sets and every reading of (o1), (o3), (o4) and (o5). See the helper's docstring for where each
parameter comes from. This is the long step.

In [ ]:
t = time.time()
rms, flagged, flagged_matlab, union, RES = il.run_all(
    sess, ONSETS, log=lambda m: print(f"{m} ({time.time() - t:.0f} s)", flush=True))
print("in vivo pipeline:", round((time.time() - t) / 60, 1), "min")

In [ ]:
print("Contact RMS over the whole recording (microvolts, as the file's unit 'volts' x 1e6):")
print(pd.DataFrame({k: (v * 1e6).round(1) for k, v in rms.items()}).T.to_string())
print()
for k in rms:
    print(f"{k}: errant (numpy quartiles) {list(np.flatnonzero(flagged[k]))}, "
          f"(MATLAB prctile) {list(np.flatnonzero(flagged_matlab[k]))}")
print("union over animals:", list(np.flatnonzero(union)), "=", int(union.sum()),
      "contacts; the paper reports 7 excluded and 25 kept")
print("sum over animals of the contacts flagged in each:", sum(int(v.sum()) for v in flagged.values()),
      "(the Methods: 'across all animals a total of seven electrode contacts were marked bad')")
print("kept contacts: union", 32 - int(union.sum()), "per mouse; own sets",
      {k: 32 - int(v.sum()) for k, v in flagged.items()})
print("median contact RMS per file (uV):", {k: round(float(np.median(v)) * 1e6, 2) for k, v in rms.items()})

In [ ]:
# The MUA band against noise. For Gaussian noise band-passed at 300 to 3000 Hz, the rate of
# downward crossings of -3 SD is about f_rms * exp(-4.5) (Rice), with f_rms the band's RMS frequency.
f = np.linspace(300, 3000, 10001)
f_rms = np.sqrt(np.trapezoid(f ** 2, f) / np.trapezoid(np.ones_like(f), f))
print("Gaussian noise: about", round(f_rms * np.exp(-4.5), 1), "crossings/s at -3 SD")
for k in rms:
    r = RES["union"][k]
    r = r.get("imaging") or r["o1a"]
    rate = r["mua_rate"][("baseline", "crossing")]
    print(f"{k}: baseline MUA crossings/s (baseline SD) median {np.median(rate):.1f}, range {rate.min():.1f} to {rate.max():.1f}")

### Readings and the values each claim uses

`KEEP`: the union of errant contacts over animals (the claims file's reading) or each animal's own.
`O1`: the alignment of interlum_6, which has no imaging: (a) the infusion start, (b) the infusion
start plus the mean delay to onset of the five imaged mice. The other readings are named where used.

In [ ]:
KEEP = ("union", "own")
O1 = ("o1a", "o1b")


def run_of(keep, subject, o1, on="imaging"):
    # on: the onset of the imaged mice, "imaging" (the claims' onset) or an alternative of the sensitivity check
    r = RES[keep][subject]
    return r[on] if on in r else r[o1]


def post_mask(r):
    return (r["centres"] >= 0) & (r["centres"] < il.POST_S)


def base_mask(r):
    return (r["centres"] >= -il.BASELINE_S) & (r["centres"] < 0)


def contact_values(keep, o1, measure, on="imaging", **kw):
    # {subject: per-contact value, pooled time-bin values (bins x contacts)}
    out = {}
    for k in GROUP:
        r = run_of(keep, k, o1, on)
        if measure in ("gamma", "gamma_hi"):
            g = r[measure][post_mask(r)]
            out[k] = (g.mean(0), g)
        elif measure == "events":
            key = (kw["rate"], kw["jk"], kw["win"])
            out[k] = (r["events"][key].astype(float), r["event_bins"][key])
        elif measure == "mua":
            m = r["mua"][(kw["span"], kw["count"])]
            out[k] = (np.nanmean(np.where(np.isfinite(m), m, np.nan), 0), m)
    return out


def summarise(vals, alternative, pct="linear"):
    a = np.concatenate([vals[k][0] for k in PLUS])
    b = np.concatenate([vals[k][0] for k in MINUS])
    pa = np.concatenate([vals[k][1].ravel() for k in PLUS])
    pb = np.concatenate([vals[k][1].ravel() for k in MINUS])
    dropped = int((~np.isfinite(a)).sum() + (~np.isfinite(b)).sum())
    a, b = a[np.isfinite(a)], b[np.isfinite(b)]
    pa, pb = pa[np.isfinite(pa)], pb[np.isfinite(pb)]
    mw = stats.mannwhitneyu(a, b, alternative=alternative)
    ks = stats.ks_2samp(pa, pb)
    m_plus = [il.prctile(vals[k][0][np.isfinite(vals[k][0])], 85, pct) for k in PLUS]
    m_minus = [il.prctile(vals[k][0][np.isfinite(vals[k][0])], 85, pct) for k in MINUS]
    mwm = stats.mannwhitneyu(m_plus, m_minus, alternative=alternative, method="exact")
    return dict(n_plus=len(a), n_minus=len(b), dropped=dropped,
                mean_plus=a.mean(), sd_plus=a.std(ddof=1), median_plus=np.median(a),
                mean_minus=b.mean(), sd_minus=b.std(ddof=1), median_minus=np.median(b),
                KS_D=ks.statistic, KS_p=ks.pvalue, n_pooled=(len(pa), len(pb)),
                U=mw.statistic, MW_p=mw.pvalue,
                mice_plus=np.round(m_plus, 3), mice_minus=np.round(m_minus, 3),
                mice_mean_plus=np.mean(m_plus), mice_sd_plus=np.std(m_plus, ddof=1),
                mice_median_plus=np.median(m_plus),
                mice_mean_minus=np.mean(m_minus), mice_sd_minus=np.std(m_minus, ddof=1),
                mice_median_minus=np.median(m_minus),
                U_mice=mwm.statistic, MW_mice_p=mwm.pvalue)


def within(x, value, tol):
    return bool(abs(x - value) <= tol + 1e-12)


def show(df, cols):
    with pd.option_context("display.float_format", "{:.4g}".format):
        print(df[cols].to_string())

In [ ]:
def table(measure, alternative, grid, on="imaging"):
    # One row per combination of readings in `grid` (list of dicts), with summarise() columns.
    rows = []
    for g in grid:
        for keep in KEEP:
            for o1 in O1:
                for pct in ("linear", "matlab"):
                    kw = {k: v for k, v in g.items()}
                    vals = contact_values(keep, o1, measure, on, **kw)
                    rows.append(dict(keep=keep, o1=o1, pct=pct, **kw, **summarise(vals, alternative, pct)))
    return pd.DataFrame(rows)


EVENT_GRID = [dict(rate=r, jk=j, win=w) for r in ("10k", "1k") for j in ("sample", "block") for w in ("onset", "infusion")]
MUA_GRID = [dict(span=sp, count=c) for sp in ("whole", "baseline") for c in ("samples", "crossing")]
GAMMA_TAB = table("gamma", "greater", [{}])
GAMMA_HI_TAB = table("gamma_hi", "greater", [{}])
EVENT_TAB = table("events", "greater", EVENT_GRID)
MUA_TAB = table("mua", "two-sided", MUA_GRID)
print("combinations: gamma", len(GAMMA_TAB), "events", len(EVENT_TAB), "MUA", len(MUA_TAB))

### The paper's own per-mouse values

Supplementary Data 7 gives, for Fig. 7e and 7f, each mouse's mean over contacts of gamma power, gamma
events and MUA, with its SEM, and, for Fig. 7d, the group mean gamma power of each contact over time.
They are compared here with the same means from this attempt (union set, the claims' onsets, (o1a)
for interlum_6, events at 10 kHz by sample after onset, MUA with the whole-recording SD and every
sample). This is a check on the data only; no outcome uses it.

In [ ]:
def sd7_table(sheet):
    rows = il.supplementary_sheet(7, sheet)
    head = [str(h) for h in rows[4][:5]]
    return pd.DataFrame([r[:5] for r in rows[5:] if r[0] is not None], columns=head).set_index("Mouse")


P7e, P7f = sd7_table("Fig. 7e"), sd7_table("Fig. 7f")
with pd.option_context("display.float_format", "{:.3f}".format):
    print("Supplementary Data 7, Fig. 7e:"); print(P7e.to_string())
    print("Supplementary Data 7, Fig. 7f:"); print(P7f.to_string())
g_ = contact_values("union", "o1a", "gamma")
e_ = contact_values("union", "o1a", "events", rate="10k", jk="sample", win="onset")
m_ = contact_values("union", "o1a", "mua", span="whole", count="samples")
OURS = pd.DataFrame({k: dict(group=GROUP[k], gamma_power_mean=g_[k][0].mean(), gamma_event_mean=e_[k][0].mean(),
                             mua_mean=np.nanmean(m_[k][0])) for k in GROUP}).T
with pd.option_context("display.float_format", "{:.3f}".format):
    print("This attempt, per mouse (mean over the 28 kept contacts):"); print(OURS.to_string())
print("Nearest paper mouse of the same group by mean gamma power:")
for k in GROUP:
    cand = P7e[[i.startswith("exp " if GROUP[k] == "Opsin (+)" else "cntrl ") and not i.endswith("mean") for i in P7e.index]]
    d = (cand["gamma power mean"].astype(float) - OURS.loc[k, "gamma_power_mean"]).abs()
    j = d.idxmin()
    print(f"  {k}: {j}, {float(cand.loc[j, 'gamma power mean']):.3f} against {OURS.loc[k, 'gamma_power_mean']:.3f} dB; "
          f"events {float(cand.loc[j, 'gamma event mean']):.1f} against {OURS.loc[k, 'gamma_event_mean']:.1f}; "
          f"MUA {float(P7f.loc[j, 'mua mean']):.1f} against {OURS.loc[k, 'mua_mean']:.1f}%")
for sheet in ("Fig. 7d L", "Fig. 7d R"):
    rows = il.supplementary_sheet(7, sheet)
    h = [x for x in rows[4] if x is not None]
    a = np.array([[np.nan if v is None else v for v in r[:len(h)]] for r in rows[5:] if isinstance(r[0], (int, float))], float)
    t, x = a[:, 0], a[:, 1:]
    bm, pm = np.nanmean(x[t < 0], 0), np.nanmean(x[(t >= 0) & (t < 1200)], 0)
    print(f"{sheet}: {x.shape[1]} contacts, numbered {h[1]} to {h[-1]} with no depth or channel; baseline mean of each "
          f"contact {bm.min():.4f} to {bm.max():.4f} dB; mean over 0 to 1200 s {pm.min():.2f} to {pm.max():.2f} dB")


The paper's per-mouse gamma power means include negative values (cntrl 2, -0.230 dB) like this attempt's, as
P against the mean baseline power gives; the Fig. 7d sheets are centred on each contact's baseline mean (within
0.005 dB of 0). Two mice match closely on mean gamma power: interlum_8 (0.349 dB) the paper's exp 2 (0.351), and
interlum_ctrl_4 (-0.234) cntrl 2 (-0.230), which the bioluminescence match also pairs with it. The paper's exp 2
is not among the imaged mice of Fig. 7b (exp 1 and exp 3), while interlum_8 has images. The other mice match no
paper mouse. The mean event counts here (8.9 to 66.1 per contact) are all below the paper's lowest mouse (129.2),
interlum_8 has 66.1 against 164.6 for exp 2, and interlum_ctrl_4 has a mean MUA change of 111.2% against 2.9% for
cntrl 2. The differences in events and MUA therefore come at least in
part from the pipeline, as the gamma power of the matched mice agrees.

## fig7b-bioluminescence-both-groups

> CTZ infusion causes a rise in bioluminescence over barrel cortex in both Opsin (+) and Opsin (-) mice.

Criterion: in every imaged mouse, the mean over 0 to 1200 s after onset exceeds the mean over the
180 s before onset by more than the upper bound of the bootstrapped 95% confidence interval of the
baseline mean (1 s frames, 10000 resamples, two-sided bound). ROI reading (o2a) cannot be run, as
the dandiset records no ROI position; (o2b), the whole frame, is run.

In [ ]:
rng = np.random.default_rng(20220701)
rows = []
for k, b in bio.items():
    on = int(onset[k])
    base, post = b[on - 180:on], b[on:on + int(il.POST_S)]
    base, post = base[~np.isnan(base)], post[~np.isnan(post)]
    ub = il.bootstrap_upper(base, rng)
    rows.append(dict(subject=k, group=GROUP[k], baseline_mean=base.mean(), bootstrap_upper=ub,
                     post_mean=post.mean(), post_s=len(post), ratio=post.mean() / base.mean(),
                     rise=bool(post.mean() > ub)))
B7 = pd.DataFrame(rows).set_index("subject")
with pd.option_context("display.float_format", "{:.4g}".format):
    print(B7.to_string())
print("mice with a rise:", int(B7.rise.sum()), "of", len(B7), "; by group",
      B7.groupby("group").rise.sum().to_dict())

**Result: reproduced, on the whole-frame reading.** All five imaged mice, 2 Opsin (+) and 3 Opsin (-), have a mean after onset above the bootstrap upper bound of their baseline mean, with ratios of 1.07 to 1.71. interlum_4 is imaged for only 808 s of the 1200 s after onset (58 blank frames removed). The ROI reading (o2a) could not be run, as the dandiset records no ROI position.

## fig7c-gamma-emerges-opsin-plus

> In Opsin (+) mice broad-band gamma oscillations emerge with CTZ, and similar changes are not seen in
> Opsin (-) mice despite bioluminescence.

Criterion: the group mean over contacts and mice of P (dB) over 30 to 100 Hz and 0 to 1200 s after
onset; reproduced when the Opsin (+) value is above 0 dB and above the Opsin (-) value, under every
reading of (o1) and (o2) (only o2b can be run). Both kept sets are shown.

In [ ]:
rows = []
for keep in KEEP:
    for o1 in O1:
        vals = contact_values(keep, o1, "gamma")
        per_mouse = {k: float(vals[k][0].mean()) for k in GROUP}
        gp = float(np.concatenate([vals[k][0] for k in PLUS]).mean())
        gm = float(np.concatenate([vals[k][0] for k in MINUS]).mean())
        rows.append(dict(keep=keep, o1=o1, opsin_plus_dB=gp, opsin_minus_dB=gm,
                         holds=bool(gp > 0 and gp > gm), **{k: round(v, 3) for k, v in per_mouse.items()}))
C7 = pd.DataFrame(rows)
with pd.option_context("display.float_format", "{:.4g}".format):
    print(C7.to_string())
print()
print("Each mouse's mean of P over 0 to 1200 s minus its own baseline mean of P (30-100 Hz, dB), union, o1a:")
rel = {}
for k in GROUP:
    r = run_of("union", k, "o1a")
    rel[k] = float(r["gamma"][post_mask(r)].mean() - r["gamma"][base_mask(r)].mean())
print({k: round(v, 3) for k, v in rel.items()})
print("group means of that difference: Opsin (+)", round(float(np.mean([rel[k] for k in PLUS])), 3),
      "; Opsin (-)", round(float(np.mean([rel[k] for k in MINUS])), 3))
# The same pipeline on white noise, which has no change: P of a 5-taper estimate against the mean power is below 0 dB.
noise = np.random.default_rng(20220703).standard_normal((1400 * 1000, 1))
cn, fn, Sn = il.mt_spectrogram(noise, 200.0, 1390.0)
gn = il.band_mean(il.db_rel_baseline(Sn, cn, 380.0), fn, il.GAMMA)
print("white noise, P over 30-100 Hz: baseline mean", round(float(gn[(cn >= 200) & (cn < 380)].mean()), 3),
      "dB, mean after 'onset'", round(float(gn[cn >= 380].mean()), 3), "dB")
del noise, Sn

fig, axs = plt.subplots(1, 2, figsize=(12, 3.5), sharey=True)
for ax, grp in zip(axs, (PLUS, MINUS)):
    runs = [run_of("union", k, "o1a") for k in grp]
    common = sorted(set.intersection(*[set(np.round(r["centres"], 1)) for r in runs]))
    spec = np.mean([r["spec_mean"][np.isin(np.round(r["centres"], 1), common)] for r in runs], 0)
    r = runs[0]
    fk = r["freqs"] <= 120
    im = ax.imshow(spec[:, fk].T, aspect="auto", origin="lower", cmap="RdBu_r", vmin=-2, vmax=2,
                   extent=[common[0], common[-1], 0, r["freqs"][fk][-1]])
    ax.set_title(f"{GROUP[grp[0]]}: mean P over contacts and mice (union, o1a)")
    ax.set_xlabel("s from onset")
    print(GROUP[grp[0]], "spectrogram range", round(float(spec[:, fk].min()), 2), "to",
          round(float(spec[:, fk].max()), 2), "dB; colour scale clipped at +/-2 dB")
axs[0].set_ylabel("Hz")
plt.colorbar(im, ax=axs, label="dB")
plt.show()

**Result: reproduced.** The Opsin (+) mean gamma power over 0 to 1200 s is 0.08 to 0.11 dB, above 0 and above the Opsin (-) mean (-0.09 dB), under both alignments of interlum_6 and both kept sets, and under the alternative onsets (section "Onset sensitivity"). The effect is small and rests mostly on one mouse: interlum_8 is at 0.31 to 0.35 dB, interlum_6 at 0.00 to 0.04 and interlum_4 at -0.05 to -0.07, while interlum_ctrl_5 is at 0.06 to 0.08, above two of the three Opsin (+) mice. P against the mean baseline power is biased below 0 dB: on white noise its baseline mean is -0.43 dB, as in every mouse here. Measured against each mouse's own baseline mean, all six mice rise, the Opsin (+) mice by 0.39 to 0.80 dB (mean 0.54) and the Opsin (-) mice by 0.20 to 0.51 dB (mean 0.34). The criterion's test holds; the statement's "not observed in Opsin (-) mice" holds only as the group comparison.


## fig7d-contacts-exceeding-baseline

> The gamma increase is seen on about half of the contacts in Opsin (+) mice (15.7 +/- 10.1 SD contacts
> per mouse) against 2.7 +/- 2.5 in Opsin (-) mice.

Criterion: a contact exceeds baseline when its mean gamma power over 0 to 1200 s after onset is above
the bootstrapped 95% confidence interval of the mean of its 180 baseline steps (10000 resamples).
Readings: band (a) 30 to 100 Hz or (b) 80 to 100 Hz; bound (a) upper end of a two-sided interval
(97.5th percentile) or (b) one-sided (95th); plus (o1). Reproduced when both means are within the
stated SDs and the Opsin (+) mean is higher, under every reading.

In [ ]:
rng = np.random.default_rng(20220704)
rng_b = np.random.default_rng(20220705)
rows, rows_b = [], []
for keep in KEEP:
    for o1 in O1:
        for band, measure in (("a 30-100", "gamma"), ("b 80-100", "gamma_hi")):
            counts = {"a two-sided": {}, "b one-sided": {}}
            for k in GROUP:
                r = run_of(keep, k, o1)
                g = r[measure]
                base, post = g[base_mask(r)], g[post_mask(r)].mean(0)
                idx = rng.integers(0, base.shape[0], size=(10_000, base.shape[0]))
                boot = base[idx].mean(1)                      # resamples x contacts
                counts["a two-sided"][k] = int((post > np.percentile(boot, 97.5, axis=0)).sum())
                counts["b one-sided"][k] = int((post > np.percentile(boot, 95, axis=0)).sum())
                blk = il.block_bootstrap_means(base, rng_b, 10)      # variant: 10 s blocks
                counts.setdefault("c block 10 s", {})[k] = int((post > np.percentile(blk, 97.5, axis=0)).sum())
            for bound, c in counts.items():
                p = np.array([c[k] for k in PLUS]); m = np.array([c[k] for k in MINUS])
                (rows_b if bound.startswith("c") else rows).append(dict(keep=keep, o1=o1, band=band, bound=bound,
                                 plus=list(p), minus=list(m), mean_plus=p.mean(), sd_plus=p.std(ddof=1),
                                 mean_minus=m.mean(), sd_minus=m.std(ddof=1),
                                 plus_ok=within(p.mean(), 15.7, 10.1), minus_ok=within(m.mean(), 2.7, 2.5),
                                 higher=bool(p.mean() > m.mean())))
D7 = pd.DataFrame(rows)
D7["holds"] = D7.plus_ok & D7.minus_ok & D7.higher
print("Mean over contacts of the baseline mean and of the post mean of P (30-100 Hz, dB), union, o1a:")
for k in GROUP:
    r = run_of("union", k, "o1a")
    print(f"  {k}: baseline {r['gamma'][base_mask(r)].mean():.3f}, post {r['gamma'][post_mask(r)].mean():.3f}")
with pd.option_context("display.float_format", "{:.3g}".format):
    print(D7.to_string())
print("readings under which the claim holds:", int(D7.holds.sum()), "of", len(D7))
D7b = pd.DataFrame(rows_b)
D7b["holds"] = D7b.plus_ok & D7b.minus_ok & D7b.higher
print()
print("Variant, moving-block bootstrap of the baseline steps (10 s blocks, two-sided 97.5th percentile):")
with pd.option_context("display.float_format", "{:.3g}".format):
    print(D7b.drop(columns="bound").to_string())
print("variant: the claim holds in", int(D7b.holds.sum()), "of", len(D7b), "; Opsin (-) mean",
      round(D7b.mean_minus.min(), 1), "to", round(D7b.mean_minus.max(), 1), "; Opsin (+) mean",
      round(D7b.mean_plus.min(), 1), "to", round(D7b.mean_plus.max(), 1))
print("kept contacts per mouse: union", 32 - int(union.sum()), "; own", {k: 32 - int(v.sum()) for k, v in flagged.items()})

In [ ]:
# The same rule on the paper's own Fig. 7d series (Supplementary Data 7): each contact's group mean
# over mice in 1 s steps from -180 to 1200 s. These are group means, not single mice, so this shows
# only what the rule gives on the paper's data at the level the paper publishes them.
def sd7_7d(sheet):
    rows = il.supplementary_sheet(7, sheet)
    h = [x for x in rows[4] if x is not None]
    a = np.array([[np.nan if v is None else v for v in r[:len(h)]] for r in rows[5:] if isinstance(r[0], (int, float))], float)
    return rows[2][0], a[:, 0], a[:, 1:]


rng_p = np.random.default_rng(20220706)
for sheet in ("Fig. 7d L", "Fig. 7d R"):
    note, t, x = sd7_7d(sheet)
    base, post = x[t < 0], x[(t >= 0) & (t < il.POST_S)].mean(0)
    boot = base[rng_p.integers(0, base.shape[0], size=(10_000, base.shape[0]))].mean(1)
    blk = il.block_bootstrap_means(base, rng_p, 10)
    print(f"{sheet}: {note}")
    print(f"  {base.shape[0]} baseline steps; contacts above the bound, of {x.shape[1]}: two-sided 97.5th",
          int((post > np.percentile(boot, 97.5, axis=0)).sum()), "| one-sided 95th",
          int((post > np.percentile(boot, 95, axis=0)).sum()), "| 10 s blocks",
          int((post > np.percentile(blk, 97.5, axis=0)).sum()))
    print("  mean over 0 to 1200 s minus baseline mean, by contact:", np.round(post - base.mean(0), 2))


**Result: not reproduced.** Under all 16 readings nearly every contact exceeds baseline in both groups: Opsin (+) 24.7 to 29.3 contacts per mouse and Opsin (-) 25.3 to 29.3, against 15.7 and 2.7 in the paper, and the Opsin (-) mean is never within 2.5 of 2.7. The baseline mean of P is about -0.43 dB in every mouse while the means after onset are -0.23 to 0.35 dB. A bootstrap over single 1 s steps treats 180 overlapping windows as independent; with a moving-block bootstrap of 10 s blocks the counts fall (Opsin (+) 19.3 to 28.7, Opsin (-) 20.3 to 29.3) and the claim still holds under none of the 8 readings. Under the alternative onsets the Opsin (-) mean is 27.3 to 29.3. The same rule on the paper's own Fig. 7d series (Supplementary Data 7, each contact's group mean over mice) counts 25 of 25 contacts above baseline in Opsin (+) and 20 of 25 in Opsin (-) (19 with 10 s blocks); the Opsin (-) group mean rises by up to 0.34 dB on the first ten contacts. These are means over three mice, and a group mean counts more contacts than single mice would (25 of 25 in Opsin (+) against the paper's 15.7 per mouse), so a per-mouse mean of 2.7 is hard to reconcile with the rule the Methods describe but not ruled out. The Methods do not say which baseline values are resampled; the claims file fixes the 1 s steps as a standard choice, and whether this should be an open reading is asked of a reader. The outcome stays not reproduced. A change to blocked (methods underspecified) was proposed; the round-3 verifier did not agree, as every reading gives the same result, and the proposal stands.


## fig7d-superficial-layers

> The gamma increase in Opsin (+) mice is localised to the more superficial and granular layers of the
> cortical depth.

Criterion: per contact, the Opsin (+) mean over mice of gamma power over 0 to 1200 s after onset,
ordered by depth from the probe map. Band (a) 80 to 100 Hz or (b) 30 to 100 Hz; split (a) upper half
against lower half of the kept contacts, (b) upper two thirds against the deepest third. Reproduced
when the upper part has the higher mean under every reading.

The probe map is the files' `Position` column. Its description is autogenerated and does not say
whether z is depth below the surface or height above the deepest contact, so both orientations are
run: "z_up" reads z = 775 as the most superficial contact, "z_down" reads z = 0 as the most superficial.

In [ ]:
pm = sess[0].probe_metadata()
same = all(s.probe_metadata()[p] == pm[p] for s in sess for p in ("table", "columns", "groups"))
print("What the files say about the probe (table, columns and groups the same in every file:", same,
      "; probe device the same in every file:", all(s.probe_metadata()["devices"]["array"] == pm["devices"]["array"] for s in sess), ")")
print("  electrode table:", pm["table"])
for c, d in pm["columns"].items():
    print(f"  column {c}: {d}")
print("  electrode groups:", pm["groups"])
print("  devices:", pm["devices"])
print("  label: (x, z) for each channel:",
      "; ".join(f"{l}: ({x:g}, {zz:g})" for l, (x, _, zz) in zip(sess[0].labels, sess[0].position)))
print()
z = sess[0].position[:, 2]
rows = []
profiles = {}
for keep in KEEP:
    for o1 in O1:
        for band, measure in (("a 80-100", "gamma_hi"), ("b 30-100", "gamma")):
            prof = np.full((len(PLUS), 32), np.nan)
            for i, k in enumerate(PLUS):
                kept = ~(union if keep == "union" else flagged[k])
                r = run_of(keep, k, o1)
                prof[i, kept] = r[measure][post_mask(r)].mean(0)
            with np.errstate(all="ignore"), warnings.catch_warnings():
                warnings.simplefilter("ignore")      # contact 18 is kept in no Opsin (+) mouse
                mean = np.nanmean(prof, 0)
            ok = ~np.isnan(mean)
            profiles[(keep, o1, band)] = mean
            for orient in ("z_up", "z_down"):
                depth = (z.max() - z) if orient == "z_up" else z      # depth below the top contact
                order = np.argsort(depth[ok])
                v = mean[ok][order]
                n = len(v)
                for split, cut in (("a halves", n // 2), ("b two thirds", int(round(2 * n / 3)))):
                    up, low = v[:cut].mean(), v[cut:].mean()
                    rows.append(dict(keep=keep, o1=o1, band=band, orientation=orient, split=split,
                                     contacts=n, upper=up, lower=low, upper_higher=bool(up > low)))
L7 = pd.DataFrame(rows)
with pd.option_context("display.float_format", "{:.3g}".format):
    print(L7.to_string())
for orient in ("z_up", "z_down"):
    sub = L7[L7.orientation == orient]
    print(orient, ": upper part higher in", int(sub.upper_higher.sum()), "of", len(sub), "readings")
print()
print("Opsin (+) mean gamma (30-100 Hz, dB) by contact, union, o1a, from z = 775 to z = 0:")
m = profiles[("union", "o1a", "b 30-100")]
print(pd.Series(m[np.argsort(-z)], index=z[np.argsort(-z)].astype(int)).round(2).to_string())

In [ ]:
# The paper's Fig. 7d profiles next to this attempt's, by contact number. Both are the rise over each
# contact's baseline mean (the paper's sheets are centred on it), union set, (o1a).
P7d = {}
for sheet in ("Fig. 7d L", "Fig. 7d R"):
    _, t, x = sd7_7d(sheet)
    P7d[sheet] = x[(t >= 0) & (t < il.POST_S)].mean(0) - x[t < 0].mean(0)
paper = pd.DataFrame({"L, Opsin (+)": P7d["Fig. 7d L"], "R, Opsin (-)": P7d["Fig. 7d R"]},
                     index=pd.RangeIndex(1, 26, name="paper contact"))
prof = {}
for band, measure in (("30-100", "gamma"), ("80-100", "gamma_hi")):
    for grp, mice in (("Opsin (+)", PLUS), ("Opsin (-)", MINUS)):
        p = np.full((len(mice), 32), np.nan)
        for i, k in enumerate(mice):
            r = run_of("union", k, "o1a")
            g = r[measure]
            p[i, ~union] = g[post_mask(r)].mean(0) - g[base_mask(r)].mean(0)
        prof[f"{grp} {band}"] = p.mean(0)
ours = pd.DataFrame(prof, index=pd.Index(sess[0].labels, name="label"))
ours.insert(0, "z", z.astype(int))
print("Paper, Supplementary Data 7 (dB over 0 to 1200 s):")
print(paper.round(2).to_string())
print()
print("This attempt, in channel order (dB over 0 to 1200 s; the union drops 4 contacts):")
print(ours.round(2).to_string())
print()
print("Spearman correlation of the rise with the contact's place in its own numbering (negative: the rise is")
print("largest at contact 1 of the paper, or at shank1-electrode1, z = 775, here):")
for c in paper:
    print(f"  paper {c}: {stats.spearmanr(np.arange(25), paper[c]).statistic:.2f}")
for c in prof:
    ok = np.isfinite(prof[c])
    print(f"  this attempt {c}: {stats.spearmanr(np.arange(32)[ok], prof[c][ok]).statistic:.2f}")


**Result: blocked (metadata insufficient).** Read with z = 775 as the most superficial contact, the upper part has the higher Opsin (+) gamma power under all 16 readings (for example 0.34 against -0.12 dB, halves, 30 to 100 Hz); read with z = 0 as the most superficial, under none. Nothing in the files says which end of the probe z is measured from: every column of the electrode table has an autogenerated description, the electrode group says only "electrode group for shank1" in barrel cortex, and the probe device only "A1x32-Poly2-5mm-50s-177". The paper's Fig. 7d source data number the 25 contacts 1 to 25 without depth or channel (data check). The profile itself is clear: with the union set and alignment (o1a), Opsin (+) gamma power is 0.26 to 0.95 dB on the eight contacts from z = 600 to 775 and at or below 0 dB on the others. Set against the paper's Fig. 7d profiles (contacts 1 to 25, rise over each contact's baseline mean), the rise here is largest at shank1-electrode1 (z = 775) and falls along the labels in both groups (Spearman rho -0.59 to -0.84), and the paper's falls from contact 1 to 25 in both panels (rho -0.56 Opsin (+), -0.92 Opsin (-)). So the paper's contact 1 is most likely the file's electrode1 end. This ties the numbering, not the depth: neither the sheets nor the files say whether that end is superficial, and the published figure's depth axis would be needed to decide. The sheets' notes give the band as 30 to 100 Hz, where the Fig. 7d legend gives 80 to 100 Hz.


## fig7e-gamma-power-electrodes

> Across electrodes, gamma power relative to baseline is higher in Opsin (+) than in Opsin (-) mice
> (mean 0.53 +/- 0.86 SD against -0.10 +/- 0.24 dB; medians 0.34 and -0.17).

Criterion: 30 to 100 Hz; KS test on all contact x time-bin x animal values of each group pooled
(1 s steps over 0 to 1200 s); one-tailed Mann-Whitney test (Opsin (+) greater) on the time-averaged
contact values. Means tolerate the stated SDs, medians 10%. Reproduced when both tests give p < 0.05
in the stated direction and the four values are within tolerance under every combination of (o1) and
(o2); partially reproduced when the tests hold and the values do not. The 80 to 100 Hz band is a variant.
The percentile reading does not enter this claim, so rows with pct = "linear" are shown.

In [ ]:
def judge(df, specs, test_cols):
    # specs: [(column, value, tolerance)]; test_cols: boolean expression columns already in df
    df = df.copy()
    for col, val, tol in specs:
        df[col + "_ok"] = [within(x, val, tol) for x in df[col]]
    df["values_ok"] = df[[c + "_ok" for c, _, _ in specs]].all(axis=1)
    df["tests_ok"] = df[test_cols].all(axis=1) if test_cols else True
    return df


SPEC_E_POWER = [("mean_plus", 0.53, 0.86), ("mean_minus", -0.10, 0.24),
                ("median_plus", 0.34, 0.034), ("median_minus", -0.17, 0.017)]
G = GAMMA_TAB[GAMMA_TAB.pct == "linear"].copy()
G["ks_sig"] = G.KS_p < 0.05
G["mw_sig"] = (G.MW_p < 0.05)
G["dir_ok"] = G.median_plus > G.median_minus      # direction by medians, as the rank test compares
G = judge(G, SPEC_E_POWER, ["ks_sig", "mw_sig", "dir_ok"])
show(G, ["keep", "o1", "n_plus", "n_minus", "mean_plus", "sd_plus", "median_plus", "mean_minus", "sd_minus",
         "median_minus", "KS_D", "KS_p", "U", "MW_p", "values_ok", "tests_ok"])
print("Opsin (+) mean also above the Opsin (-) mean in", int((G.mean_plus > G.mean_minus).sum()), "of", len(G))
print("the paper: U = 4163 with n = 75 and 75; here U is for Opsin (+) over Opsin (-)")
print("tests hold in", int(G.tests_ok.sum()), "of", len(G), "; values within tolerance in", int(G.values_ok.sum()), "of", len(G))
print("value checks:"); show(G, ["keep", "o1"] + [c + "_ok" for c, _, _ in SPEC_E_POWER])
print()
print("Variant, 80 to 100 Hz (Fig. 7d band):")
H = GAMMA_HI_TAB[GAMMA_HI_TAB.pct == "linear"]
show(H, ["keep", "o1", "mean_plus", "sd_plus", "median_plus", "mean_minus", "sd_minus", "median_minus", "KS_p", "U", "MW_p"])
print()
print("Per mouse, mean (SD) over contacts, union, o1a:")
vals = contact_values("union", "o1a", "gamma")
for k in GROUP:
    print(f"  {k} {GROUP[k]}: {vals[k][0].mean():.3f} ({vals[k][0].std(ddof=1):.3f}), contacts {len(vals[k][0])}")

**Result: partially reproduced.** Both tests hold under all four readings: the KS p is printed as 0 and the one-tailed Mann-Whitney p is 0.0009 to 0.0043, with the Opsin (+) median and mean above the Opsin (-) ones. The means are within the stated SDs (Opsin (+) 0.08 to 0.11 dB, Opsin (-) -0.09 dB), but the Opsin (+) median is -0.04 to 0.01 dB against 0.34, and the Opsin (-) median is within 10% only with each animal's own kept set (-0.165; -0.151 with the union). 28 to 31 contacts are kept per mouse, where the paper keeps 25. In the 80 to 100 Hz band of Fig. 7d the Mann-Whitney test is not significant (p 0.23 to 0.48). Under the alternative onsets the Mann-Whitney p is at most 0.0011.


## fig7e-gamma-events-electrodes

> Across electrodes, gamma events after CTZ are more numerous in Opsin (+) than in Opsin (-) mice
> (mean 213.60 +/- 139.53 SD against 139.28 +/- 72.97; medians 222 and 149).

Criterion: events per contact over the 1200 s window under (o1), (o2), (o3) (window after onset or
after infusion start) and (o4) (jackknife by sample or by 1 s block; 10 kHz or 1 kHz). KS readings:
(a) event counts per 1 s bin pooled, (b) the per-contact totals. One-tailed Mann-Whitney test on the
per-contact counts. Means tolerate the stated SDs, medians 10%.

In [ ]:
SPEC_E_EVENTS = [("mean_plus", 213.60, 139.53), ("mean_minus", 139.28, 72.97),
                 ("median_plus", 222, 22.2), ("median_minus", 149, 14.9)]
E = EVENT_TAB[EVENT_TAB.pct == "linear"].copy()
ks_tot = []
for _, row in E.iterrows():
    vals = contact_values(row.keep, row.o1, "events", rate=row.rate, jk=row.jk, win=row.win)
    a = np.concatenate([vals[k][0] for k in PLUS]); b = np.concatenate([vals[k][0] for k in MINUS])
    ks_tot.append(stats.ks_2samp(a, b).pvalue)
E["KS_p_bins"] = E.KS_p
E["KS_p_totals"] = ks_tot
E["mw_sig"] = E.MW_p < 0.05
E["ks_bins_sig"] = E.KS_p_bins < 0.05
E["ks_tot_sig"] = E.KS_p_totals < 0.05
E["dir_ok"] = E.median_plus > E.median_minus      # direction by medians, as the rank test compares
E = judge(E, SPEC_E_EVENTS, ["mw_sig", "dir_ok"])
E["tests_ok_ksa"] = E.tests_ok & E.ks_bins_sig
E["tests_ok_ksb"] = E.tests_ok & E.ks_tot_sig
show(E, ["keep", "o1", "rate", "jk", "win", "mean_plus", "sd_plus", "median_plus", "mean_minus", "sd_minus",
         "median_minus", "KS_p_bins", "KS_p_totals", "U", "MW_p", "values_ok"])
print("tests hold (KS reading a) in", int(E.tests_ok_ksa.sum()), "of", len(E),
      "; (KS reading b) in", int(E.tests_ok_ksb.sum()), "of", len(E),
      "; values within tolerance in", int(E.values_ok.sum()), "of", len(E))
print("Opsin (+) mean also above the Opsin (-) mean in", int((E.mean_plus > E.mean_minus).sum()), "of", len(E))
print("range of means: Opsin (+)", round(E.mean_plus.min(), 1), "to", round(E.mean_plus.max(), 1),
      "; Opsin (-)", round(E.mean_minus.min(), 1), "to", round(E.mean_minus.max(), 1))
print("value checks passing, by quantity:", {c: int(E[c + "_ok"].sum()) for c, _, _ in SPEC_E_EVENTS}, "of", len(E))
print()
r = run_of("union", "interlum_8", "o1a")
thr = r["thresholds"][("10k", "sample")]
print("example thresholds, interlum_8, 10 kHz, by sample: median", float(np.median(thr)) * 1e6, "uV")

**Result: partially reproduced.** Both tests hold under all 32 readings (Mann-Whitney p at most 1.1e-06; KS p at most 1.0e-07 under either KS reading), with Opsin (+) higher. The counts are far below the paper's: Opsin (+) means 28.7 to 33.6 events per contact against 213.6, Opsin (-) 10.3 to 12.5 against 139.3, and no value is within tolerance under any reading.

## fig7e-gamma-power-mice

> At the mouse level, gamma power relative to baseline is higher in Opsin (+) than in Opsin (-) mice
> (mean 1.16 +/- 1.09 SD against 0.03 +/- 0.18 dB), with every Opsin (+) mouse above every Opsin (-)
> mouse (U = 9).

Criterion: each mouse's value is the 85th percentile of its time-averaged contact values; readings
(o6) (numpy linear or MATLAB prctile) and (o1). Reproduced when U = 9 and the four values are within
tolerance under every combination; partially reproduced when U = 9 and the values are not.

In [ ]:
SPEC_M_POWER = [("mice_mean_plus", 1.16, 1.09), ("mice_mean_minus", 0.03, 0.18),
                ("mice_median_plus", 0.88, 0.088), ("mice_median_minus", 0.09, 0.009)]
GM = GAMMA_TAB.copy()
GM["U9"] = GM.U_mice == 9
GM = judge(GM, SPEC_M_POWER, ["U9"])
show(GM, ["keep", "o1", "pct", "mice_plus", "mice_minus", "mice_mean_plus", "mice_sd_plus", "mice_median_plus",
          "mice_mean_minus", "mice_sd_minus", "mice_median_minus", "U_mice", "MW_mice_p", "values_ok"])
print("U = 9 in", int(GM.U9.sum()), "of", len(GM), "; values within tolerance in", int(GM.values_ok.sum()), "of", len(GM))
print("value checks passing, by quantity:", {c: int(GM[c + "_ok"].sum()) for c, _, _ in SPEC_M_POWER}, "of", len(GM))
print("mouse order of mice_plus / mice_minus:", PLUS, MINUS)

**Result: partially reproduced.** Every Opsin (+) mouse is above every Opsin (-) mouse under all 8 readings (U = 9, exact one-tailed p = 0.05). The group means are within the stated SDs (Opsin (+) 0.63 to 0.70 dB, Opsin (-) 0.11 to 0.13 dB), but the medians are not within 10% (Opsin (+) 0.70 to 0.78 against 0.88; Opsin (-) 0.04 to 0.06 against 0.09).

## fig7e-gamma-events-mice

> At the mouse level, gamma events are more numerous in Opsin (+) than in Opsin (-) mice (mean 327.25
> +/- 45.58 SD against 199.33 +/- 36.08), with complete separation (U = 9).

Criterion: as fig7e-gamma-power-mice, on the per-contact gamma event counts, under (o1) to (o4) and (o6).

In [ ]:
SPEC_M_EVENTS = [("mice_mean_plus", 327.25, 45.58), ("mice_mean_minus", 199.33, 36.08),
                 ("mice_median_plus", 304.25, 30.4), ("mice_median_minus", 203.75, 20.4)]
EM = EVENT_TAB.copy()
EM["U9"] = EM.U_mice == 9
EM = judge(EM, SPEC_M_EVENTS, ["U9"])
show(EM, ["keep", "o1", "rate", "jk", "win", "pct", "mice_plus", "mice_minus", "mice_mean_plus",
          "mice_mean_minus", "U_mice", "MW_mice_p", "values_ok"])
print("U = 9 in", int(EM.U9.sum()), "of", len(EM), "; values within tolerance in", int(EM.values_ok.sum()), "of", len(EM))
print("U values found:", EM.U_mice.value_counts().to_dict())
print("U by percentile reading:", {p: EM[EM.pct == p].U_mice.value_counts().to_dict() for p in ("linear", "matlab")})
print("U by kept set:", {k: EM[EM.keep == k].U_mice.value_counts().to_dict() for k in KEEP})
print("range of Opsin (+) mouse means", round(EM.mice_mean_plus.min(), 1), "to", round(EM.mice_mean_plus.max(), 1),
      "; Opsin (-)", round(EM.mice_mean_minus.min(), 1), "to", round(EM.mice_mean_minus.max(), 1))

**Result: blocked (methods underspecified).** U is 9 under 36 of 64 readings and 8 or 8.5 under the others, so the readings give partially reproduced or not reproduced, and the claims file blocks the claim when they differ. U = 9 holds in 29 of 32 readings with the union of errant contacts and in 7 of 32 with each animal's own. No reading puts the values near the paper's: Opsin (+) mouse means of 39.2 to 52.4 events against 327.25, Opsin (-) 14.0 to 18.4 against 199.33. Under the alternative onsets the readings still disagree (U = 9 in 39 and 36 of 64).

## fig7f-mua-electrodes

> Across electrodes, MUA rises more from baseline in Opsin (+) than in Opsin (-) mice (mean 37.14 +/-
> 45.50 SD against 13.71 +/- 21.44 percent; medians 15.49 and 8.96).

Criterion: MUA under (o1), (o2) and (o5) (SD from the whole recording or the baseline; every sample
below threshold or downward crossings); KS on all 1 s bins over 0 to 1200 s of each group pooled;
two-tailed Mann-Whitney test on the time-averaged contact values. Means tolerate the stated SDs,
medians 10%.

In [ ]:
SPEC_E_MUA = [("mean_plus", 37.14, 45.50), ("mean_minus", 13.71, 21.44),
              ("median_plus", 15.49, 1.55), ("median_minus", 8.96, 0.90)]
M = MUA_TAB[MUA_TAB.pct == "linear"].copy()
M["ks_sig"] = M.KS_p < 0.05
M["mw_sig"] = M.MW_p < 0.05
M["dir_ok"] = M.median_plus > M.median_minus
M = judge(M, SPEC_E_MUA, ["ks_sig", "mw_sig", "dir_ok"])
show(M, ["keep", "o1", "span", "count", "dropped", "mean_plus", "sd_plus", "median_plus", "mean_minus", "sd_minus",
         "median_minus", "KS_D", "KS_p", "U", "MW_p", "values_ok", "tests_ok"])
print("Mann-Whitney p < 0.05 in", int(M.mw_sig.sum()), "of", len(M), "; KS p < 0.05 in", int(M.ks_sig.sum()),
      "; all tests hold in", int(M.tests_ok.sum()), "; values within tolerance in", int(M.values_ok.sum()))
print("Per mouse mean MUA change over contacts (%), union, o1a, SD from whole recording, samples:")
vals = contact_values("union", "o1a", "mua", span="whole", count="samples")
print({k: round(float(np.nanmean(vals[k][0])), 1) for k in GROUP})

**Result: not reproduced.** The two-tailed Mann-Whitney test finds no difference between the groups under any of the 16 readings (p 0.67 to 0.97), although the KS test on the pooled bins does (p printed as 0). The Opsin (+) mean change is 39.4 to 56.7% and the Opsin (-) 42.3 to 53.7%, and the Opsin (-) medians (20.2 to 22.7%) are above the Opsin (+) ones (13.2 to 19.5%). interlum_ctrl_4 has the largest mean change of the six mice (111.2%). This rests on the onset of interlum_ctrl_4. With its second largest rise as onset (228 s, after the infusion start, against 206 s), the baseline takes in part of the rise (19.3 against 12.1 samples per s below threshold), its mean change falls to 18.0%, and the Mann-Whitney p falls to 0.012 to 0.15. The onset matched to the paper's own series for this mouse (210 s) gives p = 0.69 to 0.92, so the onset used is the one the paper's data support.


## fig7f-mua-mice-not-significant

> At the mouse level the MUA difference is not significant (mean 77.72 +/- 47.40 SD against 27.15 +/-
> 18.38 percent; U = 9, p > 0.05, two-tailed), though Opsin (+) mice show MUA increases higher than, or
> equal to, the highest Opsin (-) values.

Criterion: two-tailed Mann-Whitney test on each mouse's 85th percentile of time-averaged contact MUA,
under (o1), (o2), (o5) and (o6). With 3 against 3 mice the test cannot reach p < 0.05, so the claim
is judged on U and the values: reproduced when U = 9 with Opsin (+) higher and the four values are
within tolerance under every combination; partially reproduced when U is 7 or 8, or U = 9 and the
values are not within tolerance.

In [ ]:
SPEC_M_MUA = [("mice_mean_plus", 77.72, 47.40), ("mice_mean_minus", 27.15, 18.38),
              ("mice_median_plus", 52.15, 5.2), ("mice_median_minus", 27.24, 2.7)]
MM = MUA_TAB.copy()
MM = judge(MM, SPEC_M_MUA, [])
show(MM, ["keep", "o1", "span", "count", "pct", "mice_plus", "mice_minus", "mice_mean_plus", "mice_mean_minus",
          "mice_median_plus", "mice_median_minus", "U_mice", "MW_mice_p", "values_ok"])
print("U values (Opsin (+) over Opsin (-)):", MM.U_mice.value_counts().to_dict(), "of", len(MM))
print("U by percentile reading:", {p: MM[MM.pct == p].U_mice.value_counts().to_dict() for p in ("linear", "matlab")})
print("highest Opsin (-) mouse value against each Opsin (+) mouse, union, o1a, whole, samples, matlab:")
row = MM[(MM.keep == "union") & (MM.o1 == "o1a") & (MM.span == "whole") & (MM["count"] == "samples") & (MM.pct == "matlab")].iloc[0]
print("  Opsin (+)", dict(zip(PLUS, row.mice_plus)), "; highest Opsin (-)", max(row.mice_minus))
print("two-tailed p >= 0.05 in", int((MM.MW_mice_p >= 0.05).sum()), "of", len(MM))
print("values within tolerance in", int(MM.values_ok.sum()), "of", len(MM),
      "; by quantity", {c: int(MM[c + "_ok"].sum()) for c, _, _ in SPEC_M_MUA})

**Result: blocked (methods underspecified).** The two-tailed p is at least 0.2 under all 32 readings, so the absence part holds, but U is 6 to 8 and never 9: U = 6 (16 readings) gives not reproduced and U = 7 or 8 (16 readings) partially reproduced under the criterion. interlum_4 (51 to 67%) is below the highest Opsin (-) mouse, interlum_ctrl_4 (105 to 128%). No reading puts the four values within tolerance; the Opsin (-) mouse means are 61.5 to 77.1% against 27.15. Under the second-rise onsets U is 9 in 28 of 32 readings, as interlum_ctrl_4's change falls (see fig7f-mua-electrodes); under the Supplementary Data 7 onsets it is 6 to 8.


## Onset sensitivity

The in vivo claims again, with each imaged mouse aligned to the alternative onsets defined in the data
check ("second" and "sd7"), under the same readings as above. This is a check on how far the outcomes
depend on the onset found from the whole-frame series; no outcome is decided here. For each onset the
table gives the statistic each claim turns on.

In [ ]:
def exceed(keep, o1, on, rng):
    # contacts per mouse whose mean gamma (30-100 Hz) after onset exceeds the two-sided 97.5th percentile
    # of the bootstrapped baseline mean (the first reading of fig7d-contacts-exceeding-baseline)
    out = {}
    for k in GROUP:
        r = run_of(keep, k, o1, on)
        base, post = r["gamma"][base_mask(r)], r["gamma"][post_mask(r)].mean(0)
        boot = base[rng.integers(0, base.shape[0], size=(10_000, base.shape[0]))].mean(1)
        out[k] = int((post > np.percentile(boot, 97.5, axis=0)).sum())
    return out


rng = np.random.default_rng(20220706)
srows = []
for on in ("imaging", "second", "sd7"):
    rise = 0
    for k, b in bio.items():
        o = int(ONSETS[k][on])
        base, post = b[o - 180:o], b[o:o + int(il.POST_S)]
        base, post = base[~np.isnan(base)], post[~np.isnan(post)]
        rise += int(post.mean() > il.bootstrap_upper(base, rng))
    c7 = []
    for keep in KEEP:
        for o1 in O1:
            v = contact_values(keep, o1, "gamma", on)
            gp = np.concatenate([v[k][0] for k in PLUS]).mean(); gm = np.concatenate([v[k][0] for k in MINUS]).mean()
            c7.append((gp, gm, gp > 0 and gp > gm))
    d7 = [exceed(keep, o1, on, rng) for keep in KEEP for o1 in O1]
    d7m = [np.mean([d[k] for k in MINUS]) for d in d7]; d7p = [np.mean([d[k] for k in PLUS]) for d in d7]
    gt, ht = table("gamma", "greater", [{}], on), None
    et = table("events", "greater", EVENT_GRID, on)
    mt = table("mua", "two-sided", MUA_GRID, on)
    gl, el, ml = gt[gt.pct == "linear"], et[et.pct == "linear"], mt[mt.pct == "linear"]
    srows.append({
        "onset": on,
        "7b mice with a rise": f"{rise} of 5",
        "7c holds": f"{sum(h for _, _, h in c7)} of {len(c7)}; (+) {min(a for a, _, _ in c7):.2f} to {max(a for a, _, _ in c7):.2f}, (-) {min(b for _, b, _ in c7):.2f} to {max(b for _, b, _ in c7):.2f} dB",
        "7d contacts/mouse": f"(+) {min(d7p):.1f} to {max(d7p):.1f}, (-) {min(d7m):.1f} to {max(d7m):.1f}",
        "7e power electrodes MW p": f"{gl.MW_p.min():.2g} to {gl.MW_p.max():.2g}; medians (+) {gl.median_plus.min():.3f} to {gl.median_plus.max():.3f}",
        "7e events electrodes MW p": f"{el.MW_p.min():.2g} to {el.MW_p.max():.2g}",
        "7e power mice U": str(gt.U_mice.value_counts().to_dict()),
        "7e events mice U": str(et.U_mice.value_counts().to_dict()),
        "7f MUA electrodes MW p": f"{ml.MW_p.min():.2g} to {ml.MW_p.max():.2g}; medians (+) {ml.median_plus.min():.1f} to {ml.median_plus.max():.1f}, (-) {ml.median_minus.min():.1f} to {ml.median_minus.max():.1f}",
        "7f MUA mice U": str(mt.U_mice.value_counts().to_dict()),
    })
SENS = pd.DataFrame(srows).set_index("onset").T
with pd.option_context("display.max_colwidth", 200):
    print(SENS.to_string())
print("onsets (file s):", {k: {on: ONSETS[k][on] for on in ("imaging", "second", "sd7")} for k in bio})
print("Per mouse under each onset, union, (o1a), MUA with the whole-recording SD and every sample: mean change over",
      "contacts (%) and baseline rate (samples below threshold per s, mean over contacts):")
for k in bio:
    print(f"  {k}:", "; ".join(f"{on} {np.nanmean(run_of('union', k, 'o1a', on)['mua'][('whole', 'samples')]):.1f}% "
                               f"at {run_of('union', k, 'o1a', on)['mua_rate'][('whole', 'samples')].mean():.1f}/s"
                               for on in ("imaging", "second", "sd7")))


The alternative onsets change no outcome except through interlum_ctrl_4 for MUA. Under both alternatives all
five imaged mice show a rise in bioluminescence, fig7c holds under every reading, the Opsin (-) mean of fig7d
stays at 27.3 to 29.3 contacts, the electrode-level gamma tests stay at p at most 0.0011, and U = 9 for gamma
power in 8 of 8 readings. The mouse-level event U still differs between readings. For MUA the second-rise onset
of interlum_ctrl_4 (228 s) puts part of its rise into the baseline (19.3 samples per s below threshold, against
12.1 to 12.2 with the other two onsets), which lowers its change from 111.2% to 18.0%; this alone moves the MUA
electrode test to p = 0.012 to 0.15 and the mouse-level U to 9 in 28 of 32 readings. The Supplementary Data 7
onset of this mouse (210 s) agrees with the one used (206 s).

## In vitro claims, on the article's source data

The per-electrode spike counts (5 s before and after each treatment) of Supplementary Data 1 to 6 of
the article, read from the publisher's site. The tests are the ones the legends name, two-tailed:
Wilcoxon matched-pairs signed-rank tests on paired columns (scipy's default, exact for small samples
without ties and a normal approximation otherwise, with zero differences dropped as in Prism) and
Mann-Whitney tests between columns (normal approximation with tie correction for these sizes, as
Prism does for large samples).

In [ ]:
def wil(before, after):
    res = stats.wilcoxon(before, after, zero_method="wilcox")
    return dict(n=len(before), median_before=np.median(before), median_after=np.median(after),
                mean_before=before.mean(), mean_after=after.mean(),
                up=int((after > before).sum()), down=int((after < before).sum()), p=res.pvalue)


def mw(x, y):
    res = stats.mannwhitneyu(x, y, alternative="two-sided")
    return dict(n_x=len(x), n_y=len(y), median_x=np.median(x), median_y=np.median(y),
                mean_x=x.mean(), mean_y=y.mean(), U=res.statistic, p=res.pvalue)


def table_print(d):
    with pd.option_context("display.float_format", "{:.4g}".format):
        print(pd.DataFrame(d).T.to_string())


SD1 = il.ladder_columns(1, "Fig. 2d")
SD2 = il.ladder_columns(2, "Fig. 3c")
SD3 = il.ladder_columns(3, "Supp Fig. 4c")
SD4c = il.ladder_columns(4, "Fig. 4c")
SD4e = il.ladder_columns(4, "Fig. 4e")
SD5 = il.ladder_columns(5, "Fig. 5c")
SD6c = il.ladder_columns(6, "Fig. 6c")
SD6f = il.ladder_columns(6, "Fig. 6f")
for name, d in [("SD1 Fig. 2d", SD1), ("SD2 Fig. 3c", SD2), ("SD3 Supp Fig. 4c", SD3), ("SD4 Fig. 4c", SD4c),
                ("SD4 Fig. 4e", SD4e), ("SD5 Fig. 5c", SD5), ("SD6 Fig. 6c", SD6c), ("SD6 Fig. 6f", SD6f)]:
    print(name, {k: (len(v), int(v.min()), int(v.max())) for k, v in d.items()})
print("(column: number of electrodes, min and max spike count)")

## fig2d-chr2-ctz-excites

> In mixed cultures with ChR2(C128S)-expressing postsynaptic neurons, blue light and CTZ both increase
> spiking, while vehicle has no consistent effect.

Criterion: Wilcoxon on 5 s counts before and after; blue light and CTZ p < 0.05 with the median higher
after, vehicle p >= 0.05; electrode counts 62, 62, 24.

In [ ]:
pre = "CHR (C128S) + luciferase 5 sec "
R2a = {t: wil(SD1[pre + "Before " + t], SD1[pre + "After " + t]) for t in ("Blue light", "CTZ", "Vehicle")}
table_print(R2a)

**Result: reproduced.** Blue light and CTZ raise the counts on all 62 electrodes (medians 1.5 to 114.5 and 0 to 82.5; Wilcoxon p = 7.6e-12 each), and vehicle does not change them (24 electrodes, p = 0.637, the paper 0.6498).

## fig2d-gtacr2-ctz-inhibits

> In mixed cultures with hGtACR2-expressing postsynaptic neurons, blue light and CTZ both decrease
> spiking, while vehicle has no consistent effect.

Criterion: as above, with the median lower after blue light and CTZ (p < 0.05 each), vehicle p >= 0.05,
49 electrodes each.

In [ ]:
pre = "hGtACR2 + luciferase 5 sec "
R2b = {t: wil(SD1[pre + "Before " + t], SD1[pre + "After " + t]) for t in ("Blue light", "CTZ", "Vehicle")}
table_print(R2b)

**Result: reproduced.** Blue light and CTZ lower the counts on all 49 electrodes (medians 24 to 9 and 42 to 8; p = 1.1e-09 each), and vehicle does not change them (p = 0.553, the paper 0.5594).

## fig3c-uncut-responses

> In connected (un-cut) co-cultures, blue light, presynaptic electrical stimulation and CTZ each
> increase spiking of the ChR2(C128S)-expressing postsynaptic neurons.

Criterion: Wilcoxon on 5 s counts before and after; each un-cut condition p < 0.05 with the median higher after.

In [ ]:
R3u = {t: wil(SD2[f"{t} 'un-cut' Before"], SD2[f"{t} 'un-cut' After"])
       for t in ("Blue light", "Electrical stimulation", "CTZ")}
table_print(R3u)

**Result: reproduced.** In un-cut co-cultures blue light (63 electrodes), stimulation (54) and CTZ (51) raise the counts on every electrode, with p from 5.2e-12 to 5.1e-10.

## fig3c-cut-abolishes-ctz

> After the connections between the two populations are cut, blue light still increases postsynaptic
> spiking, but presynaptic electrical stimulation and CTZ no longer change it.

Criterion: blue light p < 0.05 with the median higher after; electrical stimulation and CTZ p >= 0.05.

In [ ]:
R3c = {t: wil(SD2[f"{t} 'Cut' Before"], SD2[f"{t} 'Cut' After"])
       for t in ("Blue light", "Electrical stimulation", "CTZ")}
table_print(R3c)

**Result: reproduced.** After the cut, blue light still raises the counts on all 58 electrodes (p = 3.5e-11), while stimulation (32 electrodes, median 37 to 34.5, p = 0.095, the paper 0.0965) and CTZ (38 electrodes, median 65 to 60.5, p = 0.733, the paper 0.7388) do not change them.

## figs4c-blockers-silence

> The synaptic blocker cocktail silences spontaneous activity of the culture, while blue light still
> activates the opsin-expressing neurons.

Criterion: two-tailed Mann-Whitney tests, spontaneous against after SB, and after SB against blue
light; both p < 0.05, lower after SB and higher with blue light. Wilcoxon signed-rank tests as a variant.

In [ ]:
RS4 = {"spontaneous vs SB": mw(SD3["Spontaneous activity"], SD3["Synaptic blockers"]),
       "SB vs blue light": mw(SD3["Synaptic blockers"], SD3["Blue light"])}
table_print(RS4)
print("Variant, Wilcoxon signed-rank:")
table_print({"spontaneous vs SB": wil(SD3["Spontaneous activity"], SD3["Synaptic blockers"]),
             "SB vs blue light": wil(SD3["Synaptic blockers"], SD3["Blue light"])})

**Result: reproduced.** On 27 electrodes, SB lowers the counts (median 25 to 0, Mann-Whitney p = 1.8e-10) and blue light raises them after SB (0 to 29, p = 1.5e-10). The paired Wilcoxon variant gives p = 5.6e-06 for both.

## fig4c-sb-ctz-activates

> CTZ given with or immediately after the synaptic blockers increases spiking of the opsin-expressing
> neurons, while vehicle has no effect.

Criterion: two-tailed Mann-Whitney tests on the counts after treatment; SB + CTZ higher than both SB
alone and SB + vehicle at p < 0.05, and SB alone against SB + vehicle p >= 0.05.

In [ ]:
sb, ctz, veh = SD4c["Synaptic blockers (After)"], SD4c["SB + CTZ (After)"], SD4c["SB + Vehicle (fter)"]
R4c = {"SB alone vs SB + CTZ": mw(sb, ctz), "SB + CTZ vs SB + vehicle": mw(ctz, veh),
       "SB alone vs SB + vehicle": mw(sb, veh)}
table_print(R4c)
print("Before columns, for reference: medians SB", np.median(SD4c["Synaptic blockers (Before)"]),
      "SB + CTZ", np.median(SD4c["SB + CTZ (Before)"]), "SB + vehicle", np.median(SD4c["SB + Vehicle (Before)"]))

**Result: reproduced.** After treatment, SB + CTZ (median 183, 37 electrodes) is higher than SB alone (0, 27 electrodes; p = 8.0e-12) and SB + vehicle (0, 38 electrodes; p = 3.7e-14), and SB alone against SB + vehicle gives p = 0.731 (the paper 0.7305).

## fig4e-late-ctz-no-effect

> CTZ added about 20 s or more after the synaptic blockers has no effect on spiking of the
> opsin-expressing neurons.

Criterion: a claim of absence; two-tailed Mann-Whitney test of the counts after SB against after CTZ
added later (Fig. 4e(i), n = 18). Reproduced when p >= 0.05. The sheet has two columns headed
"Synaptic blockers (After)"; the first, beside "CTZ added later (After)", is panel (i).

In [ ]:
print({k: len(v) for k, v in SD4e.items()})
R4ei = {"SB (after) vs CTZ added later (after)": mw(SD4e["Synaptic blockers (After)"], SD4e["CTZ added later (After)"])}
table_print(R4ei)

**Result: reproduced.** p = 0.402 (the paper 0.4022), with medians of 3 after SB and 1.5 after CTZ added later, 18 electrodes each.

## fig4e-stim-then-ctz-activates

> During the silent phase after synaptic block, CTZ added immediately after presynaptic electrical
> stimulation increases spiking of the opsin-expressing neurons, and more than CTZ added later,
> electrical stimulation alone, or stimulation followed by vehicle.

Criterion: two-tailed Mann-Whitney tests; stimulation + CTZ higher than stimulation alone (iii), and
higher than each of CTZ added later, stimulation alone and stimulation + vehicle, all at p < 0.05.

In [ ]:
sc = SD4e["Electrical stm + CTZ (afte)"]
R4eiii = {"stim (after) vs stim + CTZ (after)": mw(SD4e["Electrical stm (After)"], sc),
          "CTZ added later vs stim + CTZ": mw(SD4e["CTZ added later (After)"], sc),
          "stim + vehicle vs stim + CTZ": mw(SD4e["Electrical stm + Vehicle (After)"], sc)}
table_print(R4eiii)
print("stim + CTZ values:", sc.astype(int).tolist())

**Result: reproduced.** Stimulation followed by CTZ (median 12, 10 electrodes) is higher than stimulation alone (0, 35 electrodes; p = 5.2e-08), CTZ added later (1.5, 18 electrodes; p = 0.00024) and stimulation followed by vehicle (0, 35 electrodes; p = 2.8e-08). The legend gives p < 0.0001 for these comparisons; the one against CTZ added later is weaker here but meets the criterion.

## fig4e-stim-alone-no-effect

> After synaptic block, presynaptic electrical stimulation alone, or followed by vehicle, does not
> change spiking of the opsin-expressing neurons.

Criterion: a claim of absence; two-tailed Mann-Whitney tests SB (after) against stimulation (after),
panel (ii), n = 35, with the second "Synaptic blockers (After)" column, and stimulation (after)
against stimulation + vehicle (after), panel (iv). Reproduced when both give p >= 0.05.

In [ ]:
R4eii = {"SB (after) vs stim (after)": mw(SD4e["Synaptic blockers (After) #2"], SD4e["Electrical stm (After)"]),
         "stim (after) vs stim + vehicle (after)": mw(SD4e["Electrical stm (After)"], SD4e["Electrical stm + Vehicle (After)"])}
table_print(R4eii)

**Result: reproduced.** SB against stimulation gives p = 1.0 (the paper > 0.9999) and stimulation against stimulation + vehicle p = 0.808 (the paper 0.8553), on 35 electrodes; all four medians are 0.

## fig5c-bont-blocks

> After 48 h of BoNT treatment, CTZ added immediately after presynaptic electrical stimulation no
> longer increases postsynaptic spiking, while blue light still does.

Criterion: two-tailed Wilcoxon tests; stimulation against stimulation + CTZ p >= 0.05, and blue light
(before against during) p < 0.05 with the median higher during light.

In [ ]:
R5 = {"stim (after) vs stim + CTZ (after)": wil(SD5["Electrical stm stm (After)"], SD5["Electrical stm stm + CTZ (After)"]),
      "blue light before vs during": wil(SD5["Blue light (Before)"], SD5["Blue light (During)"]),
      "spontaneous vs SB (after)": wil(SD5["Spontaneous activity"], SD5["Synaptic blockers (After)"])}
table_print(R5)
print("stim and stim + CTZ counts all zero:", bool((SD5["Electrical stm stm (After)"] == 0).all()),
      bool((SD5["Electrical stm stm + CTZ (After)"] == 0).all()))

**Result: reproduced.** After BoNT, stimulation followed by CTZ does not change the counts (21 electrodes, medians 0 and 0, p = 0.659, the paper 0.7173), and blue light raises them on all 21 electrodes (median 0 to 92, p = 9.5e-07).

## fig6c-green-light-abolishes

> In the presence of synaptic blockers, CTZ increases spiking of ChR2(C128S)-expressing neurons, and
> 590 nm light, which closes the step-function opsin, abolishes that spiking.

Criterion: two-tailed Wilcoxon tests. Readings: (a) SB + CTZ is spontaneous against SB + CTZ, and
green light is SB + CTZ against green light; (b) green light is spontaneous against green light.
Reproduced when SB + CTZ gives p < 0.05 with counts higher after, and green light p < 0.05 with
counts lower after, under the reading the source data's columns support. The sheet has three
columns, "Spontaneous activity", "Synaptic blockers + CTZ" and "Green light", which allow both readings.

In [ ]:
sp, sc6, gl = SD6c["Spontaneous activity"], SD6c["Synaptic blockers + CTZ"], SD6c["Green light"]
R6c = {"(a) spontaneous vs SB + CTZ": wil(sp, sc6), "(a) SB + CTZ vs green light": wil(sc6, gl),
       "(b) spontaneous vs green light": wil(sp, gl)}
table_print(R6c)

**Result: reproduced, under both readings.** SB + CTZ raises the counts over spontaneous activity (median 20 to 36, 33 electrodes up and 17 down, p = 0.0375, the paper 0.0369); this test is the same under both readings. Green light lowers the counts under reading (a), SB + CTZ against green light (all 51 electrodes, median 36 to 0, p = 5.1e-10), and under reading (b), spontaneous activity against green light (median 20 to 0, p = 0.0011).


## fig6f-nonfunctional-opsin-no-response

> With the non-functional opsin ChR2(C128S)-E97R-D253A in postsynaptic neurons, CTZ does not increase
> spiking in the presence of synaptic blockers.

Criterion: a claim of absence; two-tailed Mann-Whitney test of the counts after SB alone against
after SB + CTZ (the legend gives n = 21 and 49). Reproduced when p >= 0.05.

In [ ]:
R6f = {"SB alone (after) vs SB + CTZ (after)": mw(SD6f["Synaptic blockers (After)"], SD6f["SB + CTZ (After), non-functional opsin"])}
table_print(R6f)
print("Fig. 6f 'Synaptic blockers (After)' has", len(SD6f["Synaptic blockers (After)"]), "values (legend: n = 21);",
      "identical to Fig. 4e's first 'Synaptic blockers (After)' column:",
      bool(np.array_equal(SD6f["Synaptic blockers (After)"], SD4e["Synaptic blockers (After)"])))
print("Fig. 4c 'Synaptic blockers (Before)' identical to Supp. Fig. 4c 'Spontaneous activity':",
      bool(np.array_equal(SD4c["Synaptic blockers (Before)"], SD3["Spontaneous activity"])))
print("The Fig. 6f sheet in full:")
for k, v in SD6f.items():
    print(f"  {k} (n = {len(v)}):", v.astype(int).tolist())
print("Variant, the 21-value 'Synaptic blockers (After)' column of Fig. 5c (BoNT-treated cultures with functional",
      "ChR2(C128S), Supplementary Data 5) against the Fig. 6f SB + CTZ column:")
table_print({"Fig. 5c SB (after) vs Fig. 6f SB + CTZ (after)":
             mw(SD5["Synaptic blockers (After)"], SD6f["SB + CTZ (After), non-functional opsin"])})
print("Variant, spontaneous vs SB + CTZ (after), Wilcoxon (the legend's 'before vs after addition of SB', n = 49):")
table_print({"spontaneous vs SB + CTZ": wil(SD6f["Spontaneous activity"], SD6f["SB + CTZ (After), non-functional opsin"])})

**Result: blocked (other).** The counts this test needs, after SB alone in the cultures with the non-functional opsin, are not in the published source data. The Fig. 6f sheet's "Synaptic blockers (After)" column has 18 values where the legend gives n = 21, and it is identical to the column of Fig. 4e(i); on it the test gives p = 3.6e-10 (the paper 0.7870), SB alone with a median of 3 (18 electrodes) against SB + CTZ with a median of 0 (49 electrodes). The 21-value "Synaptic blockers (After)" column of Fig. 5c, from BoNT-treated cultures with the functional opsin, gives p = 0.534, which is not the paper's 0.7870 either. With the non-functional opsin the SB + CTZ counts are 0 on 48 of 49 electrodes and 1 on the other, which on their own show no increase.


## Summary

| Claim | Outcome |
|---|---|
| fig2d-chr2-ctz-excites | reproduced |
| fig2d-gtacr2-ctz-inhibits | reproduced |
| fig3c-uncut-responses | reproduced |
| fig3c-cut-abolishes-ctz | reproduced |
| figs4c-blockers-silence | reproduced |
| fig4c-sb-ctz-activates | reproduced |
| fig4e-late-ctz-no-effect | reproduced |
| fig4e-stim-then-ctz-activates | reproduced |
| fig4e-stim-alone-no-effect | reproduced |
| fig5c-bont-blocks | reproduced |
| figs5-bont-reduces-release | blocked: data missing |
| figs6-luciferase-in-dcvs | blocked: data missing |
| fig6c-green-light-abolishes | reproduced |
| fig6f-nonfunctional-opsin-no-response | blocked: the needed source column is not published |
| figs3-ctz-acts-postsynaptically | blocked: data missing |
| figs2-cut-disrupts-synchrony | blocked: data missing |
| fig7b-bioluminescence-both-groups | reproduced |
| fig7c-gamma-emerges-opsin-plus | reproduced |
| fig7d-contacts-exceeding-baseline | not reproduced |
| fig7d-superficial-layers | blocked: metadata insufficient |
| fig7e-gamma-power-electrodes | partially reproduced |
| fig7e-gamma-events-electrodes | partially reproduced |
| fig7e-gamma-power-mice | partially reproduced |
| fig7e-gamma-events-mice | blocked: methods underspecified |
| fig7f-mua-electrodes | not reproduced |
| fig7f-mua-mice-not-significant | blocked: methods underspecified |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning and the differences from the paper's method.

The in vitro claims are tested on the per-electrode counts the article publishes, not on recordings; they check the paper's statistics on its own source data.

How the analysis differs from the paper's:

- The bioluminescence is the mean of the whole frame; the paper's hand-placed ROI is not recorded in the dandiset. The onsets found here are 4 to 40 s from the paper's, judged by matching the series of the Opsin (-) mice to Supplementary Data 7.
- The paper's rule for errant contacts flags 1 to 3 contacts per mouse here, 4 over all mice, where the paper excludes 7 and keeps 25. Both the union and each animal's own set are run.
- The files are at 10 kHz, which the file description says were downsampled from 20 kHz; the Methods state 30 kHz, then downsampling to 10 kHz.
- interlum_6 has no imaging; it is aligned to the infusion start, or to it plus the mean delay of the other mice (43.2 s).
- Details the paper leaves open are run under each reading the claims file lists, and the outcome is the one they all give.
- As a sensitivity check, the in vivo claims are also run with two other onsets per imaged mouse. They change no outcome except for MUA, where the second-rise onset of interlum_ctrl_4 (228 s) moves the electrode test to p = 0.012 to 0.15; the paper's own series for that mouse supports the onset used (206 s here, 210 s there).
- P against the mean baseline power, as the Methods define it, is biased below 0 dB (-0.43 dB on white noise and in every mouse). The paper's per-mouse means in Supplementary Data 7 include negative values like these.
- The fig7d bootstrap over single overlapping 1 s steps is the claims file's; a 10 s block bootstrap gives the same outcome. On the paper's own Fig. 7d group series the same rule counts 20 of 25 Opsin (-) contacts above baseline, against the paper's 2.7 per mouse, which group means can inflate but makes the paper's count hard to reconcile with the rule; a change of fig7d-contacts-exceeding-baseline to blocked (methods underspecified) was proposed and the round-3 verifier did not agree, as every reading gives the same result, so the outcome stays not reproduced and the proposal stands. Whether the resampled baseline unit should be an open reading is asked of a reader.

What was found about the dataset:

- The dandiset holds only the in vivo experiment; the culture recordings, the IVIS and confocal images are not in it.
- The contact positions do not say which end of the probe is superficial, so depth profiles cannot be read.
- interlum_4 has 1111 image frames (58 of them blank, all zero), 866 s after onset, where the paper's series cover 1200 s; its images are 1024 x 1024 pixels and those of interlum_ctrl_2 684 x 697, where the Methods give 512 x 512.
- In interlum_ctrl_4 the bioluminescence rises before the recorded start of the infusion (onset 8 s before it).
- In Supplementary Data 6, the Fig. 6f column of counts after SB alone has 18 values, identical to a column of Fig. 4e, where the legend gives n = 21.
- The whole-frame series of the three Opsin (-) mice match Supplementary Data 7's cntrl 1 to 3 (r = 0.975 to 0.990); those of the two imaged Opsin (+) mice match neither exp 1 nor exp 3 as closely (r at most 0.906), and no file matches exp 1 (r at most 0.68). On mean gamma power interlum_8 matches the paper's exp 2 (0.349 against 0.351 dB), a mouse the paper did not image, and interlum_ctrl_4 matches cntrl 2.
- The signals are small for an unfiltered cortical recording: median contact RMS of 1.9 to 6.7 uV in the file's unit, volts.
